Code Explanation --

This project uses LangGraph to build a structured AI agent workflow instead of a traditional linear chatbot. LangGraph allows defining nodes and edges, enabling flexible and dynamic execution based on user intent.
The system begins with an intent detection node that classifies user input into categories such as greeting, pricing, lead, or general using rule-based logic. This ensures fast and reliable intent routing without external API dependency.
For pricing-related queries, a Retrieval-Augmented Generation (RAG) pipeline is implemented using FAISS and HuggingFace embeddings (all-MiniLM-L6-v2). The system converts knowledge base data into vector embeddings and retrieves relevant documents based on user queries, ensuring context-aware responses.
State is managed using a shared dictionary (AgentState), which persists user data such as name, email, and platform across multiple steps. This allows the agent to progressively collect missing information during the conversation.
The lead node ensures all required user details are captured, while the tool node simulates a real-world action by storing the lead data. This modular design ensures scalability, maintainability, and real-world applicability.

In [1]:
!pip install langgraph langchain-community sentence-transformers

In [2]:
import json
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Load knowledge
with open("knowledge.json") as f:
    data = json.load(f)

docs = []
for key, value in data.items():
    docs.append(Document(page_content=f"{key}: {value}"))

# Offline embeddings (no API issues)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

C:\Users\sahil\AppData\Local\Temp\ipykernel_18584\252392687.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from typing import TypedDict, Optional

class AgentState(TypedDict):
    user_input: str
    intent: str
    response: str
    name: Optional[str]
    email: Optional[str]
    platform: Optional[str]

In [14]:
def intent_node(state):
    text = state["user_input"].lower().strip()

    # HIGH INTENT FIRST
    if any(x in text for x in ["i want", "try", "buy", "subscribe", "sign up"]):
        intent = "lead"

    # THEN pricing
    elif any(x in text for x in ["price", "pricing", "cost", "plan"]):
        intent = "pricing"

    # THEN greeting
    elif text in ["hi", "hello", "hey"]:
        intent = "greeting"

    else:
        intent = "general"

    print("Detected intent:", intent)

    return {**state, "intent": intent}

In [15]:
def greeting_node(state):
    return {**state, "response": "Hey I am chatbot AI. How can I help you today?"}

In [20]:
def pricing_node(state):
    docs = retriever.invoke(state["user_input"])
    
    
    response = """
We offer two plans:

 Basic Plan – $29/month  
- 10 videos/month  
- 720p resolution  

 Pro Plan – $79/month  
- Unlimited videos  
- 4K resolution  
- AI captions 


  company policies- {
    "refund": "No refunds after 7 days",
    "support": "24/7 support available only on Pro plan"
  }

What plan do you want ? 
"""
    return {**state, "response": response}

In [21]:
def mock_lead_capture(name, email, platform):
    print(f"Lead captured: {name}, {email}, {platform}")


def route(state):
    return state["intent"]


def tool_node(state):
    # This node runs after lead is complete
    if state.get("name") and state.get("email") and state.get("platform"):
        mock_lead_capture(state["name"], state["email"], state["platform"])
        return {**state, "response": "You're all set! We'll contact you soon."}
    
    return state

In [22]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

graph.add_node("intent", intent_node)
graph.add_node("greeting", greeting_node)
graph.add_node("pricing", pricing_node)
graph.add_node("lead", lead_node)
graph.add_node("tool", tool_node)

graph.set_entry_point("intent")

graph.add_conditional_edges(
    "intent",
    route,
    {
        "greeting": "greeting",
        "pricing": "pricing",
        "lead": "lead",
        "general": "greeting"
    }
)

graph.add_edge("lead", "tool")
graph.add_edge("greeting", END)
graph.add_edge("pricing", END)
graph.add_edge("tool", END)

app = graph.compile()


In [23]:
# ---- Runtime loop ----
state = {
    "name": None,
    "email": None,
    "platform": None
}

while True:
    user_input = input("You: ")
    
    if user_input.lower() == "exit":
        print ("thanku you")
        break
    
    # Smart filling logic
    if state["name"] is None and len(user_input.split()) <= 3:
        state["name"] = user_input
    
    elif state["email"] is None and "@" in user_input:
        state["email"] = user_input
    
    elif state["platform"] is None and user_input.lower() in ["youtube", "instagram"]:
        state["platform"] = user_input
    
    state["user_input"] = user_input
    
    result = app.invoke(state)
    state.update(result)
    
    print("Agent:", result.get("response", ""))

You:  hi


Detected intent: greeting
Agent: Hey I am chatbot AI. How can I help you today?


You:  what is the price


Detected intent: pricing
Agent: 
We offer two plans:

 Basic Plan – $29/month  
- 10 videos/month  
- 720p resolution  

 Pro Plan – $79/month  
- Unlimited videos  
- 4K resolution  
- AI captions 


  company policies- {
    "refund": "No refunds after 7 days",
    "support": "24/7 support available only on Pro plan"
  }

What plan do you want ? 



You:  i want to try pro plan


...

Detected intent: pricing
Agent: 
We offer two plans:

 Basic Plan – $29/month  
- 10 videos/month  
- 720p resolution  

 Pro Plan – $79/month  
- Unlimited videos  
- 4K resolution  
- AI captions 


  company policies- {
    "refund": "No refunds after 7 days",
    "support": "24/7 support available only on Pro plan"
  }

What plan do you want ? 



You:  i want to try pro plan


Detected intent: lead
Lead captured: hi, vishal@gmail.com, instagram
Agent: You're all set! We'll contact you soon.


You:  exit


thanku you
